In [1]:
import json
import time
from pathlib import Path
from typing import Any, Dict, List

import pandas as pd
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed

# ==========================================
# CONFIGURAÇÕES
# ==========================================
URL_BASE = "https://pncp.gov.br/api/consulta/v1/contratacoes/proposta"
DATA_FINAL = "20271231"   # Data futura para pegar todas as abertas
TAMANHO_PAGINA = 50       # Limite máximo da rota
MAX_WORKERS = 5           # Evite exagerar para não tomar bloqueio
TIMEOUT = 15
MAX_TENTATIVAS = 4
PAUSA_ENTRE_TENTATIVAS = 2  # segundos

PASTA_SAIDA = Path("licitacoes")
ARQUIVO_CSV = PASTA_SAIDA / "licitacoes_abertas.csv"
ARQUIVO_JSON = PASTA_SAIDA / "licitacoes_abertas.json"


# ==========================================
# FUNÇÕES AUXILIARES
# ==========================================
def garantir_pasta_saida() -> None:
    """Cria a pasta de saída se ela não existir."""
    PASTA_SAIDA.mkdir(parents=True, exist_ok=True)


def montar_parametros(pagina: int) -> Dict[str, Any]:
    """Monta os parâmetros da requisição."""
    return {
        "dataFinal": DATA_FINAL,
        "pagina": pagina,
        "tamanhoPagina": TAMANHO_PAGINA,
    }


def fazer_requisicao(pagina: int) -> Dict[str, Any]:
    """
    Faz a requisição para uma página com retry.
    Retorna o JSON da resposta ou um dicionário vazio em caso de falha.
    """
    parametros = montar_parametros(pagina)

    for tentativa in range(1, MAX_TENTATIVAS + 1):
        try:
            print(f"[Página {pagina}] Tentativa {tentativa}/{MAX_TENTATIVAS}...")
            resposta = requests.get(URL_BASE, params=parametros, timeout=TIMEOUT)

            if resposta.status_code == 200:
                return resposta.json()

            print(
                f"[Página {pagina}] Status inesperado: {resposta.status_code}"
            )

        except requests.RequestException as erro:
            print(f"[Página {pagina}] Erro: {erro}")

        if tentativa < MAX_TENTATIVAS:
            time.sleep(PAUSA_ENTRE_TENTATIVAS)

    print(f"[Página {pagina}] Falhou após {MAX_TENTATIVAS} tentativas.")
    return {}


def buscar_pagina(pagina: int) -> List[Dict[str, Any]]:
    """Baixa uma página e retorna apenas a lista de dados."""
    resposta_json = fazer_requisicao(pagina)
    return resposta_json.get("data", [])


def descobrir_total_paginas() -> Dict[str, Any]:
    """
    Faz a primeira requisição para descobrir total de páginas
    e já aproveita os dados da página 1.
    """
    print("Descobrindo total de páginas...")
    dados_iniciais = fazer_requisicao(1)

    if not dados_iniciais:
        raise RuntimeError(
            "Não foi possível obter a página 1 da API. "
            "Verifique conexão, disponibilidade da API ou parâmetros."
        )

    total_paginas = dados_iniciais.get("totalPaginas", 1)
    itens_pagina_1 = dados_iniciais.get("data", [])

    print(f"Total de páginas encontradas: {total_paginas}")
    print(f"Itens na página 1: {len(itens_pagina_1)}")

    return {
        "total_paginas": total_paginas,
        "itens_pagina_1": itens_pagina_1,
    }


def baixar_paginas_em_paralelo(total_paginas: int) -> List[Dict[str, Any]]:
    """
    Baixa páginas da 2 até a última em paralelo.
    """
    if total_paginas <= 1:
        return []

    todos_itens = []
    paginas = range(2, total_paginas + 1)

    print(f"Iniciando download paralelo com {MAX_WORKERS} workers...")

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futuros = {executor.submit(buscar_pagina, pagina): pagina for pagina in paginas}

        for futuro in as_completed(futuros):
            pagina = futuros[futuro]
            try:
                itens = futuro.result()
                print(f"[Página {pagina}] {len(itens)} itens baixados.")
                todos_itens.extend(itens)
            except Exception as erro:
                print(f"[Página {pagina}] Erro inesperado no processamento: {erro}")

    return todos_itens


def salvar_csv(df: pd.DataFrame) -> None:
    """Salva os dados em CSV."""
    df.to_csv(ARQUIVO_CSV, index=False, encoding="utf-8-sig")
    print(f"CSV salvo em: {ARQUIVO_CSV}")


def salvar_json(registros: List[Dict[str, Any]]) -> None:
    """Salva os dados em JSON formatado."""
    with open(ARQUIVO_JSON, "w", encoding="utf-8") as arquivo:
        json.dump(registros, arquivo, ensure_ascii=False, indent=2)
    print(f"JSON salvo em: {ARQUIVO_JSON}")


def executar_coleta() -> None:
    """Fluxo principal de execução."""
    garantir_pasta_saida()

    descoberta = descobrir_total_paginas()
    total_paginas = descoberta["total_paginas"]
    todos_itens = descoberta["itens_pagina_1"]

    itens_restantes = baixar_paginas_em_paralelo(total_paginas)
    todos_itens.extend(itens_restantes)

    df = pd.DataFrame(todos_itens)

    print(f"\n✅ Coleta finalizada! Total de registros: {len(df)}")

    salvar_csv(df)
    salvar_json(todos_itens)


# ==========================================
# EXECUÇÃO
# ==========================================
if __name__ == "__main__":
    executar_coleta()

Descobrindo total de páginas...
[Página 1] Tentativa 1/4...
Total de páginas encontradas: 697
Itens na página 1: 50
Iniciando download paralelo com 5 workers...
[Página 2] Tentativa 1/4...
[Página 3] Tentativa 1/4...
[Página 4] Tentativa 1/4...
[Página 5] Tentativa 1/4...
[Página 6] Tentativa 1/4...
[Página 6] Status inesperado: 204
[Página 7] Tentativa 1/4...[Página 2] 50 itens baixados.

[Página 8] Tentativa 1/4...
[Página 3] 50 itens baixados.
[Página 6] Tentativa 2/4...
[Página 9] Tentativa 1/4...[Página 4] 50 itens baixados.

[Página 10] Tentativa 1/4...
[Página 5] 50 itens baixados.
[Página 11] Tentativa 1/4...[Página 8] 50 itens baixados.

[Página 12] Tentativa 1/4...[Página 7] 50 itens baixados.

[Página 13] Tentativa 1/4...[Página 11] 50 itens baixados.

[Página 14] Tentativa 1/4...[Página 9] 50 itens baixados.

[Página 15] Tentativa 1/4...[Página 10] 50 itens baixados.

[Página 16] Tentativa 1/4...[Página 12] 50 itens baixados.

[Página 17] Tentativa 1/4...
[Página 13] 50 ite

In [4]:
# Filtrar licitações por dataAberturaProposta entre 2026-03-20 e hoje
from pathlib import Path
import json
import pandas as pd
from datetime import datetime
from IPython.display import display

# usa ARQUIVO_JSON existente no notebook se disponível, caso contrário cria Path padrão
ARQUIVO_JSON = globals().get("ARQUIVO_JSON", Path("licitacoes") / "licitacoes_abertas.json")

if not Path(ARQUIVO_JSON).exists():
    print(f"Arquivo não encontrado: {ARQUIVO_JSON}")
else:
    with open(ARQUIVO_JSON, encoding="utf-8") as f:
        registros = json.load(f)

    df = pd.DataFrame(registros)

    if "dataAberturaProposta" not in df.columns:
        print("Coluna 'dataAberturaProposta' não encontrada no JSON.")
    else:
        # converter para datetime, ignorando valores inválidos
        df["dataAberturaProposta"] = pd.to_datetime(df["dataAberturaProposta"], errors="coerce")

        inicio = pd.to_datetime("2026-03-21")
        fim = pd.to_datetime(pd.Timestamp.now().normalize())

        mask = (df["dataAberturaProposta"] >= inicio) & (df["dataAberturaProposta"] <= fim)
        df_filtrado = df.loc[mask].reset_index(drop=True)

        print(f"Registros entre {inicio.date()} e {fim.date()}: {len(df_filtrado)}")
        # Exibir as primeiras linhas para verificação rápida
        display(df_filtrado.head(50))

        # salvar resultado em CSV na pasta 'licitacoes'
        out_dir = Path("licitacoes")
        out_dir.mkdir(parents=True, exist_ok=True)
        out_csv = out_dir / f"licitacoes_filtradas_20260320_ate_{fim.date()}.csv"
        df_filtrado.to_csv(out_csv, index=False, encoding="utf-8-sig")
        print(f"CSV salvo em: {out_csv}")


Registros entre 2026-03-21 e 2026-03-22: 32


,dataAtualizacao,orgaoEntidade,anoCompra,sequencialCompra,numeroCompra,processo,objetoCompra,orgaoSubRogado,unidadeOrgao,unidadeSubRogada,...,modoDisputaId,valorTotalEstimado,modalidadeNome,modoDisputaNome,fontesOrcamentarias,situacaoCompraId,situacaoCompraNome,usuarioNome,tipoInstrumentoConvocatorioCodigo,tipoInstrumentoConvocatorioNome
0,2026-03-20T14:41:53,"{'cnpj': '10565000000192', 'razaoSocial': 'MUN...",2026,71,3101.0004,45179,CONTRATAÇÃO DE EMPRESA QUALIFICADA PARA EXECUÇ...,None,"{'ufNome': 'Pernambuco', 'codigoUnidade': '18'...",None,...,4,0.00,Dispensa,Dispensa Com Disputa,[],1,Divulgada no PNCP,Prefeitura da Cidade do Recife,2,Aviso de Contratação Direta
1,2026-03-20T15:37:47,"{'cnpj': '17005000000187', 'razaoSocial': 'MUN...",2026,9,000006,000029/2026,CREDENCIAMENTO POR CHAMADA PÚBLICA PARA FUTURA...,None,"{'ufNome': 'Minas Gerais', 'codigoIbge': '3127...",None,...,5,0.00,Credenciamento,Não se aplica,[],1,Divulgada no PNCP,E & L PRODUCOES DE SOFTWARE LTDA,4,Edital de Chamamento Público
2,2026-03-20T08:37:06,"{'cnpj': '01613167000190', 'razaoSocial': 'MUN...",2026,43,024,040/2026,"Locação de brinquedos infláveis, cama elástica...",None,"{'ufNome': 'Paraná', 'codigoUnidade': '1', 'uf...",None,...,3,199571.90,Pregão - Eletrônico,Aberto-Fechado,[],1,Divulgada no PNCP,Bolsa Nacional De Compras - BNC,1,Edital
3,2026-03-20T07:58:41,"{'cnpj': '18668368000198', 'razaoSocial': 'MUN...",2026,86,000038,000067/2026,AQUISIÇÃO DE MATERIAL DE EXPEDIENTE PARA USO E...,None,"{'ufNome': 'Minas Gerais', 'codigoUnidade': '4...",None,...,1,172906.50,Pregão - Eletrônico,Aberto,[],1,Divulgada no PNCP,Licitar Digital - Plataforma de Licitações Online,1,Edital
4,2026-03-20T07:54:10,"{'cnpj': '18668368000198', 'razaoSocial': 'MUN...",2026,85,000038,000067/2026,AQUISIÇÃO DE MATERIAL DE EXPEDIENTE PARA USO E...,None,"{'ufNome': 'Minas Gerais', 'codigoUnidade': '1...",None,...,1,172907.00,Pregão - Eletrônico,Aberto,[],1,Divulgada no PNCP,E & L PRODUCOES DE SOFTWARE LTDA,1,Edital
5,2026-03-20T09:02:42,"{'cnpj': '95640736000130', 'razaoSocial': 'MUN...",2026,22,13 | Processo 24,13,REGISTRO DE PRECOS PARA FUTURA E EVENTUAL CONT...,None,"{'ufNome': 'Paraná', 'codigoIbge': '4128625', ...",None,...,1,2685000.00,Pregão - Eletrônico,Aberto,[],1,Divulgada no PNCP,Governançabrasil Tecnologia e Gestão em Serviços,1,Edital
6,2026-03-20T10:03:34,"{'cnpj': '50853555000154', 'razaoSocial': 'SER...",2026,65,2026/000024,2026/002978,CONTRATAÇÃO DE EMPRESA PARA FORNECIMENTO DE MA...,None,"{'ufNome': 'São Paulo', 'codigoUnidade': '203'...",None,...,1,15120.80,Pregão - Eletrônico,Aberto,[],1,Divulgada no PNCP,Bolsa Nacional De Compras - BNC,1,Edital
7,2026-03-20T10:30:02,"{'cnpj': '75845503000167', 'razaoSocial': 'MUN...",2026,28,18,33,Solicitação de abertura de Processo Licitatóri...,None,"{'ufNome': 'Paraná', 'codigoIbge': '4105102', ...",None,...,1,218799.60,Pregão - Eletrônico,Aberto,[],1,Divulgada no PNCP,Equiplano Sistemas LTDA / Equiplano Sistemas,1,Edital
8,2026-03-20T11:04:47,"{'cnpj': '01609780000134', 'razaoSocial': 'MUN...",2026,4,4,57/2026,Registro de Preços para eventual aquisição par...,None,"{'ufNome': 'Minas Gerais', 'codigoUnidade': '1...",None,...,1,1413930.85,Pregão - Eletrônico,Aberto,[],1,Divulgada no PNCP,Licitar Digital - Plataforma de Licitações Online,1,Edital
9,2026-03-20T12:12:02,"{'cnpj': '50853555000154', 'razaoSocial': 'SER...",2026,66,2026/000023,2026/003055,CONTRATAÇÃO DE EMPRESA PARA FORNECIMENTO DE ÓL...,None,"{'ufNome': 'São Paulo', 'codigoUnidade': '203'...",None,...,1,7.62,Pregão - Eletrônico,Aberto,[],1,Divulgada no PNCP,Bolsa Nacional De Compras - BNC,1,Edital


CSV salvo em: licitacoes/licitacoes_filtradas_20260320_ate_2026-03-22.csv


In [5]:
from __future__ import annotations

import json
import math
import time
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Tuple
from concurrent.futures import ThreadPoolExecutor, as_completed

import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry


# =========================================================
# CONFIG
# =========================================================
@dataclass(frozen=True)
class PncpConfig:
    url_base: str = "https://pncp.gov.br/api/consulta/v1/contratacoes/publicacao"
    data_inicial: str = "20260315"
    data_final: str = "20260321"

    modalidades: Tuple[int, ...] = tuple(range(1, 14))

    tamanho_pagina: int = 50
    timeout: int = 20

    max_workers_modalidades: int = 4
    max_workers_paginas: int = 8

    max_tentativas: int = 4
    pausa_base_retry: float = 1.5

    pasta_saida: Path = Path("licitacoes")
    nome_arquivo_base: str = "publicacoes_semana"

    user_agent: str = "Mozilla/5.0 (compatible; PNCPCollector/1.0)"


# =========================================================
# DOMAIN / DTO
# =========================================================
@dataclass(frozen=True)
class PageRequest:
    modalidade: int
    pagina: int


@dataclass
class PageResult:
    modalidade: int
    pagina: int
    items: List[Dict[str, Any]] = field(default_factory=list)
    total_paginas: int = 1
    paginas_restantes: int = 0
    sucesso: bool = True
    erro: Optional[str] = None


# =========================================================
# LOG
# =========================================================
class Logger:
    @staticmethod
    def info(message: str) -> None:
        print(f"[INFO] {message}")

    @staticmethod
    def warning(message: str) -> None:
        print(f"[WARN] {message}")

    @staticmethod
    def error(message: str) -> None:
        print(f"[ERROR] {message}")


# =========================================================
# HTTP CLIENT
# =========================================================
class PncpHttpClient:
    def __init__(self, config: PncpConfig) -> None:
        self.config = config
        self.session = self._build_session()

    def _build_session(self) -> requests.Session:
        session = requests.Session()

        retry_strategy = Retry(
            total=0,  # retry manual no método, para maior controle
            connect=0,
            read=0,
            redirect=0,
            status=0,
        )

        adapter = HTTPAdapter(
            pool_connections=50,
            pool_maxsize=50,
            max_retries=retry_strategy,
        )

        session.mount("http://", adapter)
        session.mount("https://", adapter)
        session.headers.update(
            {
                "Accept": "application/json",
                "User-Agent": self.config.user_agent,
            }
        )
        return session

    def fetch_page(self, request: PageRequest) -> PageResult:
        params = {
            "dataInicial": self.config.data_inicial,
            "dataFinal": self.config.data_final,
            "codigoModalidadeContratacao": request.modalidade,
            "pagina": request.pagina,
            "tamanhoPagina": self.config.tamanho_pagina,
        }

        for tentativa in range(1, self.config.max_tentativas + 1):
            try:
                response = self.session.get(
                    self.config.url_base,
                    params=params,
                    timeout=self.config.timeout,
                )

                if response.status_code == 200:
                    payload = response.json()
                    items = payload.get("data", []) or []
                    total_paginas = self._extract_total_paginas(payload)
                    paginas_restantes = int(payload.get("paginasRestantes", 0) or 0)

                    return PageResult(
                        modalidade=request.modalidade,
                        pagina=request.pagina,
                        items=items,
                        total_paginas=total_paginas,
                        paginas_restantes=paginas_restantes,
                        sucesso=True,
                    )

                Logger.warning(
                    f"Modalidade {request.modalidade}, página {request.pagina}: "
                    f"status {response.status_code} na tentativa {tentativa}"
                )

            except requests.RequestException as exc:
                Logger.warning(
                    f"Modalidade {request.modalidade}, página {request.pagina}: "
                    f"erro de rede na tentativa {tentativa}: {exc}"
                )
            except ValueError as exc:
                Logger.warning(
                    f"Modalidade {request.modalidade}, página {request.pagina}: "
                    f"erro ao interpretar JSON na tentativa {tentativa}: {exc}"
                )

            if tentativa < self.config.max_tentativas:
                backoff = self.config.pausa_base_retry * (2 ** (tentativa - 1))
                time.sleep(backoff)

        return PageResult(
            modalidade=request.modalidade,
            pagina=request.pagina,
            sucesso=False,
            erro=(
                f"Falha após {self.config.max_tentativas} tentativas "
                f"(modalidade={request.modalidade}, pagina={request.pagina})"
            ),
        )

    @staticmethod
    def _extract_total_paginas(payload: Dict[str, Any]) -> int:
        total_paginas = payload.get("totalPaginas")
        if total_paginas is not None:
            try:
                return max(1, int(total_paginas))
            except (TypeError, ValueError):
                pass

        total_registros = payload.get("totalRegistros")
        tamanho_pagina = payload.get("tamanhoPagina")
        if total_registros is not None and tamanho_pagina:
            try:
                total_registros_int = int(total_registros)
                tamanho_pagina_int = int(tamanho_pagina)
                if tamanho_pagina_int > 0:
                    return max(1, math.ceil(total_registros_int / tamanho_pagina_int))
            except (TypeError, ValueError):
                pass

        paginas_restantes = payload.get("paginasRestantes")
        pagina_atual = payload.get("numeroPagina", 1)
        try:
            return max(1, int(pagina_atual) + int(paginas_restantes))
        except (TypeError, ValueError):
            return 1


# =========================================================
# PAGINATION
# =========================================================
class PncpPaginator:
    def __init__(self, client: PncpHttpClient) -> None:
        self.client = client

    def discover_first_page_and_total(self, modalidade: int) -> PageResult:
        return self.client.fetch_page(PageRequest(modalidade=modalidade, pagina=1))


# =========================================================
# DEDUP
# =========================================================
class PncpDeduplicator:
    @staticmethod
    def deduplicate(records: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
        seen: set[str] = set()
        unique_records: List[Dict[str, Any]] = []

        for record in records:
            record_id = PncpDeduplicator._build_key(record)
            if record_id in seen:
                continue
            seen.add(record_id)
            unique_records.append(record)

        return unique_records

    @staticmethod
    def _build_key(record: Dict[str, Any]) -> str:
        preferred_keys = [
            "numeroControlePNCP",
            "id",
            "sequencialCompra",
            "numeroCompra",
        ]

        for key in preferred_keys:
            value = record.get(key)
            if value not in (None, ""):
                return f"{key}:{value}"

        serialized = json.dumps(record, sort_keys=True, ensure_ascii=False)
        return f"hash:{serialized}"


# =========================================================
# STORAGE
# =========================================================
class PncpStorageService:
    def __init__(self, config: PncpConfig) -> None:
        self.config = config
        self.config.pasta_saida.mkdir(parents=True, exist_ok=True)

    def save_all(self, records: List[Dict[str, Any]]) -> Tuple[Path, Path]:
        base_name = (
            f"{self.config.nome_arquivo_base}_"
            f"{self.config.data_inicial}_a_{self.config.data_final}"
        )

        csv_path = self.config.pasta_saida / f"{base_name}.csv"
        json_path = self.config.pasta_saida / f"{base_name}.json"

        df = pd.DataFrame(records)
        df.to_csv(csv_path, index=False, encoding="utf-8-sig")

        with open(json_path, "w", encoding="utf-8") as file:
            json.dump(records, file, ensure_ascii=False, indent=2)

        return csv_path, json_path


# =========================================================
# COLLECTOR
# =========================================================
class PncpCollector:
    def __init__(
        self,
        config: PncpConfig,
        paginator: PncpPaginator,
        client: PncpHttpClient,
        deduplicator: PncpDeduplicator,
    ) -> None:
        self.config = config
        self.paginator = paginator
        self.client = client
        self.deduplicator = deduplicator

    def collect(self) -> List[Dict[str, Any]]:
        Logger.info("Iniciando descoberta das primeiras páginas por modalidade...")
        first_pages = self._fetch_first_pages_concurrently()

        all_records: List[Dict[str, Any]] = []
        pending_requests: List[PageRequest] = []

        for result in first_pages:
            if not result.sucesso:
                Logger.error(result.erro or f"Falha na modalidade {result.modalidade}")
                continue

            all_records.extend(result.items)

            Logger.info(
                f"Modalidade {result.modalidade}: "
                f"página 1 com {len(result.items)} itens, "
                f"total de páginas estimado = {result.total_paginas}"
            )

            if result.total_paginas > 1:
                pending_requests.extend(
                    PageRequest(modalidade=result.modalidade, pagina=page_number)
                    for page_number in range(2, result.total_paginas + 1)
                )

        if pending_requests:
            Logger.info(
                f"Iniciando coleta simultânea das páginas restantes "
                f"({len(pending_requests)} páginas)..."
            )
            remaining_records = self._fetch_remaining_pages_concurrently(pending_requests)
            all_records.extend(remaining_records)

        Logger.info(f"Total bruto coletado: {len(all_records)} registros")
        deduplicated = self.deduplicator.deduplicate(all_records)
        Logger.info(f"Total após deduplicação: {len(deduplicated)} registros")

        return deduplicated

    def _fetch_first_pages_concurrently(self) -> List[PageResult]:
        results: List[PageResult] = []

        with ThreadPoolExecutor(max_workers=self.config.max_workers_modalidades) as executor:
            future_map = {
                executor.submit(self.paginator.discover_first_page_and_total, modalidade): modalidade
                for modalidade in self.config.modalidades
            }

            for future in as_completed(future_map):
                modalidade = future_map[future]
                try:
                    result = future.result()
                    results.append(result)
                except Exception as exc:
                    results.append(
                        PageResult(
                            modalidade=modalidade,
                            pagina=1,
                            sucesso=False,
                            erro=f"Erro inesperado na modalidade {modalidade}: {exc}",
                        )
                    )

        results.sort(key=lambda item: item.modalidade)
        return results

    def _fetch_remaining_pages_concurrently(
        self,
        requests_to_fetch: List[PageRequest],
    ) -> List[Dict[str, Any]]:
        all_records: List[Dict[str, Any]] = []

        with ThreadPoolExecutor(max_workers=self.config.max_workers_paginas) as executor:
            future_map = {
                executor.submit(self.client.fetch_page, request): request
                for request in requests_to_fetch
            }

            for future in as_completed(future_map):
                request = future_map[future]
                try:
                    result = future.result()
                    if result.sucesso:
                        Logger.info(
                            f"Modalidade {result.modalidade}, página {result.pagina}: "
                            f"{len(result.items)} itens"
                        )
                        all_records.extend(result.items)
                    else:
                        Logger.error(
                            result.erro
                            or f"Falha na modalidade {request.modalidade}, página {request.pagina}"
                        )
                except Exception as exc:
                    Logger.error(
                        f"Erro inesperado na modalidade {request.modalidade}, "
                        f"página {request.pagina}: {exc}"
                    )

        return all_records


# =========================================================
# APPLICATION
# =========================================================
class PncpApplication:
    def __init__(self, config: PncpConfig) -> None:
        self.config = config
        self.client = PncpHttpClient(config)
        self.paginator = PncpPaginator(self.client)
        self.deduplicator = PncpDeduplicator()
        self.collector = PncpCollector(
            config=config,
            paginator=self.paginator,
            client=self.client,
            deduplicator=self.deduplicator,
        )
        self.storage = PncpStorageService(config)

    def run(self) -> None:
        records = self.collector.collect()

        if not records:
            Logger.warning("Nenhum dado encontrado para o período informado.")
            return

        csv_path, json_path = self.storage.save_all(records)

        Logger.info("Coleta finalizada com sucesso.")
        Logger.info(f"Total final de registros: {len(records)}")
        Logger.info(f"CSV salvo em: {csv_path}")
        Logger.info(f"JSON salvo em: {json_path}")


# =========================================================
# MAIN
# =========================================================
if __name__ == "__main__":
    config = PncpConfig(
        data_inicial="20260315",
        data_final="20260321",
        max_workers_modalidades=4,
        max_workers_paginas=8,
        tamanho_pagina=50,
    )
    app = PncpApplication(config)
    app.run()

[INFO] Iniciando descoberta das primeiras páginas por modalidade...
[WARN] Modalidade 2, página 1: status 204 na tentativa 1
[WARN] Modalidade 2, página 1: status 204 na tentativa 1
[WARN] Modalidade 2, página 1: status 204 na tentativa 2
[WARN] Modalidade 2, página 1: status 204 na tentativa 2
[WARN] Modalidade 2, página 1: status 204 na tentativa 3
[WARN] Modalidade 2, página 1: status 204 na tentativa 3
[WARN] Modalidade 2, página 1: status 204 na tentativa 4
[INFO] Modalidade 1: página 1 com 25 itens, total de páginas estimado = 1
[ERROR] Falha após 4 tentativas (modalidade=2, pagina=1)
[INFO] Modalidade 3: página 1 com 9 itens, total de páginas estimado = 1
[INFO] Modalidade 4: página 1 com 50 itens, total de páginas estimado = 29
[INFO] Modalidade 5: página 1 com 50 itens, total de páginas estimado = 2
[INFO] Modalidade 6: página 1 com 50 itens, total de páginas estimado = 162
[INFO] Modalidade 7: página 1 com 50 itens, total de páginas estimado = 5
[INFO] Modalidade 8: página 1 